# A2 — Priprava podatkov: čas dostave naročila

Iz več tabel prvega dne pripravimo **eno vrstico na naročilo** in širši nabor
značilk. Napoved izdelamo ob oddaji naročila. Cilj `dni_dostave` je pretečeni čas
od oddaje do dostave, v dnevih z decimalnim delom.

To je retrospektiven učni prikaz: izvoz nima zgodovine sprememb katalogov ali
časa zajema vsake vrednosti. Predpostavimo, da so bile postavke, cene, katalog in
lokacije dostopni ob naročilu. Ne ustvarjamo zgodovinskih povprečij iz prihodnjih naročil.

**Odločitev:** zdaj izvajamo pravila, ki ne zahtevajo učenja iz cele zbirke: čiščenje,
smiselne agregate, razmerja, fiksne časovne oznake in kazalnike manjkanja.
Neznane številke ostanejo `NaN`, neznane kategorije dobijo `NEZNANO`.

Čista tabela še ni končna numerična matrika za model. Mediano, skaliranje,
one-hot kodiranje, združevanje redkih kategorij in izbor značilk bomo določili
**po delitvi**, samo na učnih podatkih. Tukaj ničesar ne delimo in ne učimo modelov.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

KOREN = Path.cwd().resolve()
while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
    KOREN = KOREN.parent
if not (KOREN / "utils/orodja.py").is_file():
    raise FileNotFoundError("Odprite zvezek znotraj mape projekta.")
if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
from utils.orodja import poskrbi_za_podatke

DATA = poskrbi_za_podatke()
IZHOD = DATA / "pripravljeno"
IZHOD.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 15)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})

## 1. Naložimo potrebne tabele in preverimo ključe
Uporabimo naročila, postavke, izdelke, kupce, prodajalce in prevod kategorij.
Plačil ne dodamo, ker izvoz ne potrjuje njihovega stanja ob oddaji naročila.
Ocen kupcev ne uporabimo, ker nastanejo pozneje. Podrobne tabele najprej
agregiramo; različne vrstice istega ključa niso »navadni duplikati«.

In [ ]:
mapa = DATA / "trgovina"
def preberi(ime):
    pot = mapa / ime
    stolpci = pd.read_csv(pot, nrows=0).columns
    tipi = {c: "string" for c in stolpci if c.endswith("_id") or "zip_code_prefix" in c}
    return pd.read_csv(pot, dtype=tipi)

narocila = preberi("olist_orders_dataset.csv")
postavke = preberi("olist_order_items_dataset.csv")
izdelki = preberi("olist_products_dataset.csv")
kupci = preberi("olist_customers_dataset.csv")
prodajalci = preberi("olist_sellers_dataset.csv")
prevod = preberi("product_category_name_translation.csv")
tabele = {"naročila": (narocila, ["order_id"]), "postavke": (postavke, ["order_id", "order_item_id"]),
          "izdelki": (izdelki, ["product_id"]), "kupci": (kupci, ["customer_id"]),
          "prodajalci": (prodajalci, ["seller_id"]), "prevod": (prevod, ["product_category_name"])}
pregled = []
for ime, (df, kljuci) in tabele.items():
    podvojenih = int(df.duplicated().sum())
    df.drop_duplicates(inplace=True)
    assert df[kljuci].notna().all().all(), f"Manjka ključ: {ime}"
    assert not df.duplicated(kljuci).any(), f"Različne vrstice istega ključa: {ime}"
    pregled.append({"tabela": ime, "vrstic": len(df), "odstranjene enake vrstice": podvojenih})
display(pd.DataFrame(pregled))

## 2. Uredimo tipe in neveljavne vrednosti
Masa in mere morajo biti pozitivne; ničelno maso obravnavamo kot neznano.
Ničelna poštnina pa je lahko brezplačna dostava, zato ostane 0. Negativne zneske
postavk označimo kot neznane; izvoz nima ločenega pomena vračila na tej ravni.
Veljavnih dragih ali težkih naročil ne režemo po percentilih.

In [ ]:
for df, cols in [(izdelki, ["product_category_name"]), (prevod, ["product_category_name", "product_category_name_english"]),
                 (kupci, ["customer_state"]), (prodajalci, ["seller_state"])]:
    for c in cols:
        df[c] = df[c].astype("string").str.strip().replace("", pd.NA)
for df, c in [(kupci, "customer_state"), (prodajalci, "seller_state")]:
    df[c] = df[c].str.upper()

popravki = []
for df, cols, strogo_pozitivno in [
    (postavke, ["price", "freight_value"], False),
    (izdelki, ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"], True),
    (izdelki, ["product_name_lenght", "product_description_lenght", "product_photos_qty"], False),
]:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        napaka = df[c].le(0) if strogo_pozitivno else df[c].lt(0)
        popravki.append({"stolpec": c, "neveljavnih": int(napaka.sum())})
        df[c] = df[c].mask(napaka)
for c in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    narocila[c] = pd.to_datetime(narocila[c], errors="coerce")
display(pd.DataFrame(popravki))

## 3. Obogatimo postavke brez množenja vrstic
Vsaka postavka dobi lastnosti izdelka in lokaciji kupca ter prodajalca.
`many_to_one` preveri pričakovano povezavo. Kategorijo prevedemo; če prevoda
ni, ohranimo izvirno ime. Podatek o razdalji iz teh tabel ni na voljo, zato ga
ne izmišljamo; uporabimo lokaciji in primerjavo zveznih držav.

In [ ]:
izdelki = izdelki.merge(prevod, on="product_category_name", how="left", validate="many_to_one")
izdelki["kategorija"] = izdelki["product_category_name_english"].combine_first(izdelki["product_category_name"])
izdelki["prostornina_cm3"] = izdelki["product_length_cm"] * izdelki["product_height_cm"] * izdelki["product_width_cm"]
izdelki["najdaljsa_stranica_cm"] = izdelki[["product_length_cm", "product_height_cm", "product_width_cm"]].max(axis=1, skipna=False)

lokacija_kupca = narocila[["order_id", "customer_id"]].merge(
    kupci[["customer_id", "customer_state"]], on="customer_id", how="left", validate="many_to_one")
p = (postavke.merge(izdelki[["product_id", "kategorija", "product_weight_g", "prostornina_cm3",
            "najdaljsa_stranica_cm", "product_name_lenght", "product_description_lenght", "product_photos_qty"]],
            on="product_id", how="left", validate="many_to_one")
     .merge(prodajalci[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one")
     .merge(lokacija_kupca[["order_id", "customer_state"]], on="order_id", how="left", validate="many_to_one"))
assert len(p) == len(postavke)
for c in ["product_weight_g", "prostornina_cm3", "kategorija", "seller_state", "price", "freight_value"]:
    p[c + "_manjka"] = p[c].isna().astype("int8")
lokaciji_znani = p["customer_state"].notna() & p["seller_state"].notna()
p["ista_drzava"] = p["customer_state"].eq(p["seller_state"]).astype("Float64").where(lokaciji_znani).astype(float)
print("Postavk pred povezovanjem in po njem:", len(postavke), len(p))

## 4. Agregiramo na eno vrstico na naročilo
Pri vsotah cen, poštnine, mase in prostornine zahtevamo vse znane postavke:
če ena manjka, je celotna vsota neznana. Delne vsote ne predstavimo kot polne.
Povprečja in maksimumi opisujejo znane vrednosti; dodamo tudi deleže manjkanja.

Za reprezentativno kategorijo in prodajalca izberemo **najdražjo postavko**;
pri enaki ceni odloči številka postavke. To ni nujno najpogostejša kategorija.

In [ ]:
def popolna_vsota(s):
    return s.sum(min_count=len(s))

# Ime: (izvor, agregacija, skupina, opis)
agregati = {
    "st_postavk": ("order_item_id", "size", "kosarica", "Število postavk v naročilu."),
    "st_izdelkov": ("product_id", "nunique", "kosarica", "Število različnih izdelkov."),
    "st_prodajalcev": ("seller_id", "nunique", "lokacija", "Število različnih prodajalcev."),
    "vrednost": ("price", popolna_vsota, "kosarica", "Vsota cen vseh postavk brez poštnine, BRL; NaN ob delnem manjkanju."),
    "postnina": ("freight_value", popolna_vsota, "kosarica", "Vsota poštnine vseh postavk, BRL; NaN ob delnem manjkanju."),
    "cena_postavke_min": ("price", "min", "kosarica", "Najnižja znana cena postavke, BRL."),
    "cena_postavke_max": ("price", "max", "kosarica", "Najvišja znana cena postavke, BRL."),
    "cena_postavke_povp": ("price", "mean", "kosarica", "Povprečje znanih cen postavk, BRL."),
    "razpon_cen": ("price", lambda s: s.max() - s.min(), "kosarica", "Razlika med najvišjo in najnižjo znano ceno, BRL."),
    "teza_g": ("product_weight_g", popolna_vsota, "izdelki", "Skupna masa vseh postavk, g; NaN ob delnem manjkanju."),
    "teza_povp_g": ("product_weight_g", "mean", "izdelki", "Povprečna masa postavke z znano maso, g."),
    "teza_max_g": ("product_weight_g", "max", "izdelki", "Največja znana masa postavke, g."),
    "prostornina_cm3": ("prostornina_cm3", popolna_vsota, "izdelki", "Vsota prostornin izdelkov, cm³; ni prostornina transportnega paketa."),
    "prostornina_max_cm3": ("prostornina_cm3", "max", "izdelki", "Največja znana prostornina postavke, cm³."),
    "najdaljsa_stranica_cm": ("najdaljsa_stranica_cm", "max", "izdelki", "Najdaljša znana stranica med izdelki naročila, cm."),
    "dolzina_imena_povp": ("product_name_lenght", "mean", "katalog", "Povprečna dolžina znanih imen izdelkov, znaki."),
    "dolzina_opisa_povp": ("product_description_lenght", "mean", "katalog", "Povprečna dolžina znanih opisov izdelkov, znaki."),
    "stevilo_fotografij_povp": ("product_photos_qty", "mean", "katalog", "Povprečno število fotografij pri znanih izdelkih."),
    "st_znanih_kategorij": ("kategorija", "nunique", "katalog", "Število različnih znanih kategorij; neznanih ne štejemo."),
    "st_znanih_drzav_prodajalcev": ("seller_state", "nunique", "lokacija", "Število različnih znanih zveznih držav prodajalcev."),
    "delez_postavk_ista_drzava": ("ista_drzava", "mean", "lokacija", "Delež postavk iz iste države kot kupec, med postavkami z znanima lokacijama."),
    "delez_postavk_brez_teze": ("product_weight_g_manjka", "mean", "manjkanje", "Delež postavk brez veljavne mase."),
    "delez_postavk_brez_prostornine": ("prostornina_cm3_manjka", "mean", "manjkanje", "Delež postavk brez veljavnih mer."),
    "delez_postavk_brez_kategorije": ("kategorija_manjka", "mean", "manjkanje", "Delež postavk brez znane kategorije."),
    "delez_postavk_brez_lokacije_prodajalca": ("seller_state_manjka", "mean", "manjkanje", "Delež postavk brez znane države prodajalca."),
    "delez_postavk_brez_cene": ("price_manjka", "mean", "manjkanje", "Delež postavk brez veljavne cene."),
    "delez_postavk_brez_postnine": ("freight_value_manjka", "mean", "manjkanje", "Delež postavk brez veljavne poštnine."),
}
po_narocilu = p.groupby("order_id").agg(**{ime: pd.NamedAgg(column=vir, aggfunc=agg)
    for ime, (vir, agg, _, _) in agregati.items()}).reset_index()
glavna = (p.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True], na_position="last")
          .drop_duplicates("order_id")[["order_id", "kategorija", "seller_state"]])
po_narocilu = po_narocilu.merge(glavna, on="order_id", how="left", validate="one_to_one")
assert po_narocilu["order_id"].is_unique
assert po_narocilu["st_postavk"].sum() == len(postavke)
# Pri naročilih z vsemi znanimi cenami se morajo vsote ohraniti.
popolna = po_narocilu.loc[po_narocilu["delez_postavk_brez_cene"].eq(0), "order_id"]
np.testing.assert_allclose(po_narocilu.loc[po_narocilu["order_id"].isin(popolna), "vrednost"].sum(),
                           p.loc[p["order_id"].isin(popolna), "price"].sum())
print("Postavke:", len(p), "→ naročila z vsaj eno postavko:", len(po_narocilu))

## 5. Določimo cilj in obseg zbirke
Cilj poznamo pri dostavljenih naročilih z veljavnima datumoma in nenegativnim
trajanjem. Naročil brez znanega cilja ali brez postavk ne uporabimo za učenje;
ne pripišemo jim mediane časa dostave. Zelo dolge veljavne dostave ostanejo.
Rezultati bodo zato veljali za naročila z opazovano dostavo, ne avtomatično za
stornirana ali še nedostavljena naročila.

Datum nakupa in datum, ko je cilj postal znan, ohranimo **samo kot metapodatka**
za naslednje poglavje o časovnih delitvah. Nobeden ni samodejno vhod v model.

Histogram pokaže porazdelitev ciljnega časa dostave v izbrani zbirki.
Dolge veljavne dostave ohranimo; ciljnega stolpca ne uporabimo kot značilke.


In [ ]:
tabela = (narocila[["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_delivered_customer_date",
                    "order_estimated_delivery_date"]]
    .merge(kupci[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left", validate="many_to_one")
    .merge(po_narocilu, on="order_id", how="left", validate="one_to_one"))
assert len(tabela) == len(narocila) and tabela["order_id"].is_unique
# total_seconds ohrani tudi del dneva; .dt.days bi ga odrezal.
tabela["dni_dostave"] = (tabela["order_delivered_customer_date"] - tabela["order_purchase_timestamp"]).dt.total_seconds() / 86400
# Rok, ki ga trgovina pokaže kupcu ob nakupu: znan je ob oddaji naročila, zato je lahko značilka.
tabela["obljubljeni_dnevi"] = (tabela["order_estimated_delivery_date"] - tabela["order_purchase_timestamp"]).dt.total_seconds() / 86400
pravila = {
    "dostavljeno naročilo": tabela["order_status"].eq("delivered"),
    "znana datuma": tabela[["order_purchase_timestamp", "order_delivered_customer_date"]].notna().all(axis=1),
    "nenegativno trajanje": tabela["dni_dostave"].ge(0),
    "vsaj ena postavka": tabela["st_postavk"].gt(0),
    "znan kupec za poznejše skupinsko preverjanje": tabela["customer_unique_id"].notna(),
}
obdrzi = pd.Series(True, index=tabela.index)
izlocitve = []
for razlog, pogoj in pravila.items():
    prej = int(obdrzi.sum()); obdrzi &= pogoj
    izlocitve.append({"pravilo": razlog, "izločeno v tem koraku": prej - int(obdrzi.sum()), "ostane": int(obdrzi.sum())})
display(pd.DataFrame(izlocitve))
tabela = tabela.loc[obdrzi].drop(columns=["customer_id", "order_status", "order_estimated_delivery_date"]).rename(columns={
    "order_purchase_timestamp": "datum_nakupa", "order_delivered_customer_date": "datum_znanega_izida",
    "customer_state": "zvezna_drzava", "seller_state": "drzava_glavnega_prodajalca"}).copy()
assert tabela["order_id"].is_unique

fig, ax = plt.subplots(figsize=(6, 3.5))
tabela["dni_dostave"].plot.hist(bins=50, ax=ax, color="#2993AD")
ax.set(title="Cilj: pretečeni čas dostave", xlabel="Dnevi", ylabel="Naročila")
fig.tight_layout(); plt.show()


In [ ]:
# Vnaprej določena pravila; nobeno ne računa parametrov iz vseh vrstic.
def razmerje(stevec, imenovalec):
    return stevec / imenovalec.where(imenovalec.gt(0))

def kazalnik(pogoj, znano):
    return pogoj.astype(float).where(znano)

# Slovar značilk bomo izvozili skupaj s tabelo.
opisi = {}
skupine = {}
def dodaj(ime, vrednosti, skupina, opis, vir):
    tabela[ime] = vrednosti
    skupine.setdefault(skupina, []).append(ime)
    opisi[ime] = {"opis": opis, "vir": vir, "skupina": skupina}

In [ ]:
for ime, (vir, _, skupina, opis) in agregati.items():
    dodaj(ime, tabela[ime], skupina, opis, vir + " → agregat po order_id")
for ime, skupina, opis, vir in [
    ("kategorija", "katalog", "Kategorija najdražje postavke; pri enaki ceni manjši order_item_id.", "product_category_name"),
    ("zvezna_drzava", "lokacija", "Brazilska zvezna država kupca pri naročilu.", "customer_state"),
    ("drzava_glavnega_prodajalca", "lokacija", "Zvezna država prodajalca najdražje postavke.", "seller_state"),
]:
    dodaj(ime, tabela[ime].astype("string"), skupina, opis, vir)

## 6. Dodamo časovne značilke, razmerja in transformacije
Uporabimo čas oddaje naročila, ne časa dostave. Sinus in kosinus ohranita
cikličnost: nedelja je blizu ponedeljka, december blizu januarja.
Razmerja imajo zaščiten imenovalec, logaritmi pa ne potrebujejo učenja parametrov.
Nobena značilka ne uporablja ciljnega časa dostave.

Histograma primerjata izvirno `vrednost` košarice in `log1p_vrednost = log(1 + vrednost)`.
Logaritem stisne velike vrednosti in ohrani ničle. Izvirni stolpec ostane;
korist dodatne značilke bomo preverili pozneje na validaciji.


In [ ]:
cas = tabela["datum_nakupa"]
for ime, vrednost, opis in [
    ("leto_nakupa", cas.dt.year, "Koledarsko leto oddaje naročila."),
    ("mesec_nakupa", cas.dt.month, "Koledarski mesec, 1–12."),
    ("dan_v_mesecu", cas.dt.day, "Dan v mesecu, 1–31."),
    ("dan_v_tednu", cas.dt.dayofweek, "Ponedeljek=0, nedelja=6."),
    ("ura_nakupa", cas.dt.hour, "Ura naročila v zapisanem času izvoza, 0–23."),
    ("vikend", cas.dt.dayofweek.ge(5).astype("int8"), "1 = sobota ali nedelja."),
]:
    dodaj(ime, vrednost, "cas_narocila", opis, "order_purchase_timestamp")
for ime, vrednost, perioda in [("mesec", tabela["mesec_nakupa"] - 1, 12),
                              ("dan_tedna", tabela["dan_v_tednu"], 7), ("ura", tabela["ura_nakupa"], 24)]:
    dodaj(ime + "_sin", np.sin(2 * np.pi * vrednost / perioda), "cas_narocila", "Sinus ciklične časovne značilke.", "order_purchase_timestamp")
    dodaj(ime + "_cos", np.cos(2 * np.pi * vrednost / perioda), "cas_narocila", "Kosinus ciklične časovne značilke.", "order_purchase_timestamp")

dodaj("obljubljeni_dnevi", tabela["obljubljeni_dnevi"], "obljuba",
      "Rok dostave, ki ga trgovina obljubi kupcu ob nakupu, v dnevih od oddaje naročila.", "order_estimated_delivery_date")
for ime, stevec, imenovalec, opis in [
    ("postnina_na_vrednost", "postnina", "vrednost", "Razmerje poštnine in vrednosti blaga; ni delež celotnega računa."),
    ("postnina_na_postavko", "postnina", "st_postavk", "Povprečna poštnina na postavko, BRL."),
    ("delez_najdrazje_postavke", "cena_postavke_max", "vrednost", "Delež najdražje postavke v skupni vrednosti blaga."),
    ("masa_na_prostornino", "teza_g", "prostornina_cm3", "Razmerje skupne mase in vsote prostornin izdelkov, g/cm³."),
]:
    dodaj(ime, razmerje(tabela[stevec], tabela[imenovalec]), "izpeljane", opis, f"{stevec}, {imenovalec}")
dodaj("vec_prodajalcev", tabela["st_prodajalcev"].gt(1).astype("int8"), "izpeljane", "1 = v naročilu sodeluje več prodajalcev.", "st_prodajalcev")
dodaj("vec_postavk", tabela["st_postavk"].gt(1).astype("int8"), "izpeljane", "1 = naročilo vsebuje več postavk.", "st_postavk")
for c in ["vrednost", "postnina", "teza_g", "prostornina_cm3"]:
    dodaj("log1p_" + c, np.log1p(tabela[c]), "transformirane", "Naravni logaritem (1 + vrednost).", c)

fig, osi = plt.subplots(1, 2, figsize=(11, 3.5))
tabela["vrednost"].dropna().plot.hist(bins=45, ax=osi[0], color="#2993AD")
osi[0].set(title="Izvirna vrednost košarice", xlabel="Vrednost v BRL", ylabel="Naročila")
tabela["log1p_vrednost"].dropna().plot.hist(bins=45, ax=osi[1], color="#205781")
osi[1].set(title="Dodatna logaritemska značilka", xlabel="log(1 + vrednost v BRL)", ylabel="Naročila")
fig.tight_layout(); plt.show()


## 7. Označimo in pregledamo manjkanje
Ohranimo manjkajoče številke in neznane kategorije označimo s stalno oznako.
Kazalnike ustvarimo pred nadomeščanjem kategorij, da ohranimo informacijo o manjkanju.
Numeričnih vrednosti še ne nadomeščamo z mediano; to naredimo šele po delitvi,
z mediano učne množice.

Najprej pogledamo deleže manjkajočih vrednosti. Nato primerjamo znane in
manjkajoče vrednosti `teza_g` s kazalnikom `teza_g_manjka`: 1 pomeni manjkajoč
podatek, 0 pa znanega. Kazalnik ostane na voljo tudi po poznejšem nadomeščanju.


In [ ]:
for c in ["teza_g", "prostornina_cm3", "vrednost", "postnina", "kategorija", "zvezna_drzava", "drzava_glavnega_prodajalca"]:
    dodaj(c + "_manjka", tabela[c].isna().astype("int8"), "manjkanje", "1 = podatek manjka ali je neveljaven.", c)
for c in ["kategorija", "zvezna_drzava", "drzava_glavnega_prodajalca"]:
    tabela[c] = tabela[c].fillna("NEZNANO")

display(tabela.isna().mean().mul(100).sort_values(ascending=False).head(10).rename("manjka (%)").to_frame())

# Primeri znane in manjkajoče mase ter pripadajočega kazalnika.
primeri = pd.concat([
    tabela.loc[tabela["teza_g"].notna()].head(3),
    tabela.loc[tabela["teza_g"].isna()].head(3),
])
display(primeri[["teza_g", "teza_g_manjka"]])


## 8. Določimo nabore in preverimo mejo med značilkami ter metapodatki
Osnovni nabor postopoma razširimo z lokacijo, katalogom, časom in izpeljavami.
Shema izrecno našteje dovoljene vhode. `datum_znanega_izida` ostane zunaj X:
v naslednjem poglavju bo pomagal izločiti učne cilje, ki ob časovni meji še niso bili znani.

In [ ]:
IME, KLJUC, CILJ = "olist", "order_id", "dni_dostave"
META = ["order_id", "customer_unique_id", "datum_nakupa", "datum_znanega_izida"]
DATUMI = ["datum_nakupa", "datum_znanega_izida"]
for c, opis, vir in [
    ("order_id", "ID naročila; ključ, ne značilka.", "order_id"),
    ("customer_unique_id", "ID ponovnih nakupov istega kupca; za skupinsko delitev, ne značilka.", "customer_unique_id"),
    ("datum_nakupa", "Čas napovedi; metapodatek za časovno delitev.", "order_purchase_timestamp"),
    ("datum_znanega_izida", "Čas dostave; najzgodnejši trenutek znanega cilja, nikoli značilka.", "order_delivered_customer_date"),
    ("dni_dostave", "Pretečeni čas od naročila do dostave v dnevih, brez zaokroževanja na cele dneve.", "(order_delivered_customer_date - order_purchase_timestamp) / 1 dan"),
]:
    opisi[c] = {"opis": opis, "vir": vir, "skupina": "cilj" if c == CILJ else "metapodatki"}
vse = [c for v in skupine.values() for c in v]
osnovni = ["vrednost", "postnina", "st_postavk", "teza_g", "zvezna_drzava"]
nabori = {
    "osnovni": osnovni,
    "kosarica_in_lokacija": list(dict.fromkeys(osnovni + skupine["kosarica"] + skupine["lokacija"])),
    "izvorni_agregati": [c for g in ["kosarica", "izdelki", "katalog", "lokacija", "cas_narocila"] for c in skupine[g]],
    "brez_kataloga": [c for c in vse if c not in skupine["katalog"] and c not in ["kategorija_manjka", "delez_postavk_brez_kategorije"]],
    "vsi": vse,
}
IZLOCENI = ["order_status", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date",
    "shipping_limit_date", "review_score", "review_comment_message", "plačila"]
OPOMBE = ["Ni delitve, imputacije z naučeno statistiko, kodiranja, skaliranja ali modela.",
    "Napoved ob oddaji naročila; vhodne informacije o katalogu, košarici in lokaciji predpostavimo kot takrat znane.",
    "Vključena so dostavljena naročila z znanim ciljem, kupcem in vsaj eno postavko; nedostavljeni niso dobili izmišljenega y.",
    "datum_znanega_izida je metapodatek za poznejšo časovno delitev, nikoli značilka.",
    "Čas izida lahko v izvornem sistemu postane dostopen pozneje od same dostave; izvoz tega zamika ne vsebuje.",
    "Numerične NaN ohranimo, kategorije dopolnimo s stalno oznako NEZNANO.",
    "Vsota z vsaj eno neznano postavko ostane neznana; povprečja opisujejo znane postavke."]
assert tabela[CILJ].ge(0).all()
assert tabela["datum_znanega_izida"].ge(tabela["datum_nakupa"]).all()
np.testing.assert_allclose(tabela[CILJ], (tabela["datum_znanega_izida"] - tabela["datum_nakupa"]).dt.total_seconds() / 86400)
for c in [c for c in tabela if c.startswith("delez_")]:
    assert tabela[c].dropna().between(0, 1).all(), c
assert not set(IZLOCENI) & set(vse)
display(pd.DataFrame({"nabor": list(nabori), "značilk": [len(v) for v in nabori.values()]}))

## 9. Preverimo in shranimo čisto tabelo
CSV vsebuje tudi cilj ter ID-je in datume za poznejše preverjanje modelov.
V X vključimo samo značilke iz sheme. Ponovni zagon prepiše samo pripravljeni
CSV in JSON za Olist, ne izvornih datotek.

In [ ]:
znacilke = [c for stolpci in skupine.values() for c in stolpci]
assert len(znacilke) == len(set(znacilke))
assert set(tabela.columns) == set(META + [CILJ] + znacilke)
assert not set(META + [CILJ]) & set(znacilke)
assert tabela[KLJUC].notna().all() and tabela[KLJUC].is_unique
assert tabela[CILJ].notna().all()
assert not np.isinf(tabela.select_dtypes(include="number").to_numpy(dtype=float)).any()
assert all(set(v) <= set(znacilke) for v in nabori.values())

kategoricne = [c for c in znacilke if not pd.api.types.is_numeric_dtype(tabela[c])]
numericne = [c for c in znacilke if c not in kategoricne]
tabela = tabela[META + [CILJ] + znacilke].sort_values(KLJUC).reset_index(drop=True)
slovar = []
for c in tabela:
    vloga = "cilj" if c == CILJ else ("metapodatek" if c in META else "znacilka")
    slovar.append({"stolpec": c, "vloga": vloga, "dtype": str(tabela[c].dtype),
        "manjkajocih": int(tabela[c].isna().sum()), **opisi[c]})
shema = {
    "razlicica": 1, "zbirka": IME, "datoteka": f"{IME}_modeliranje.csv",
    "kljuc": KLJUC, "cilj": CILJ, "metapodatki": META, "datumski_stolpci": DATUMI,
    "numericne_znacilke": numericne, "kategoricne_znacilke": kategoricne,
    "znacilke": znacilke, "skupine_znacilk": skupine, "nabori_znacilk": nabori,
    "stolpci": slovar, "vrstic": len(tabela), "stevilo_znacilk": len(znacilke),
    "izloceni_vhodi": IZLOCENI, "opombe": OPOMBE,
}
pot_csv = IZHOD / shema["datoteka"]
pot_json = IZHOD / f"{IME}_shema.json"
tabela.to_csv(pot_csv, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
pot_json.write_text(json.dumps(shema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# Preverimo, da CSV in shema ohranita tipe, ključe in manjkajoče vrednosti.
tipi = {r["stolpec"]: r["dtype"] for r in slovar if r["stolpec"] not in DATUMI}
prebrana = pd.read_csv(pot_csv, dtype=tipi, parse_dates=DATUMI,
    keep_default_na=False, na_values=[""])
pd.testing.assert_frame_equal(tabela, prebrana, check_exact=False, rtol=1e-10, atol=1e-10)
print(f"Shranjeno: {len(tabela):,} vrstic, {len(znacilke)} značilk + cilj in metapodatki.")
print(pot_csv.relative_to(KOREN))
print(pot_json.relative_to(KOREN))
display(pd.DataFrame([{"skupina": k, "značilk": len(v)} for k, v in skupine.items()]))
display(tabela.head(3))

## Predaja naslednjemu poglavju
Pripravljena je ena tabela na ravni naročila, brez delitve. Pri naslednjem koraku
bomo upoštevali čas napovedi, razpoložljivost učnih ciljev ter po potrebi skupine kupcev.
Median, kodiranja in skaliranja še nismo naučili.

In [ ]:
# To je vhod za naslednji sklop. Še vedno ni delitve ali modela.
shema = json.loads(pot_json.read_text(encoding="utf-8"))
X = prebrana[shema["znacilke"]].copy()
y = prebrana[shema["cilj"]].copy()
metapodatki = prebrana[shema["metapodatki"]].copy()
print("X:", X.shape, "y:", y.shape, "metapodatki:", metapodatki.shape)